<a href="https://colab.research.google.com/github/Rimpy481/Q-A-Chatbot/blob/main/Q_A_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Install libraries


In [1]:
!pip install -q --upgrade transformers accelerate sentence-transformers faiss-cpu
print("✅ Done! Ab Cell 2 run kar0")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 32.9 MB/s eta 0:00:00
✅ Done! Ab Cell 2 run kar0


Cell 2 — Restart kernel


In [ ]:
import os
os.kill(os.getpid(), 9)
# "Session crashed" aayega — normal hai. Cell 3 se continue karo.

Cell 3 — Imports


In [1]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

print("✅ All imports done!")

✅ All imports done!


Cell 4 — Knowledge base banao


In [2]:
# Yahan apna knowledge data daalo
# Agent inhi articles mein se jawab dhundega

knowledge_base = [
    {
        "id": 1,
        "topic": "python",
        "question": "What is Python?",
        "answer": "Python is a high-level, easy-to-learn programming language. "
                  "It is widely used in data science, web development, automation, and AI. "
                  "Python was created by Guido van Rossum and first released in 1991."
    },
    {
        "id": 2,
        "topic": "python",
        "question": "What are Python libraries?",
        "answer": "Python libraries are collections of pre-written code that help you perform tasks easily. "
                  "Popular ones include NumPy for math, Pandas for data, Matplotlib for graphs, "
                  "and TensorFlow and PyTorch for machine learning."
    },
    {
        "id": 3,
        "topic": "machine learning",
        "question": "What is machine learning?",
        "answer": "Machine learning is a type of artificial intelligence where computers learn from data "
                  "without being explicitly programmed. It uses algorithms to find patterns in data "
                  "and make predictions or decisions automatically."
    },
    {
        "id": 4,
        "topic": "machine learning",
        "question": "What is supervised learning?",
        "answer": "Supervised learning is a machine learning technique where the model is trained on "
                  "labeled data. The algorithm learns from input-output pairs and then makes "
                  "predictions on new unseen data. Examples include spam detection and image classification."
    },
    {
        "id": 5,
        "topic": "deep learning",
        "question": "What is deep learning?",
        "answer": "Deep learning is a subset of machine learning that uses neural networks with many layers. "
                  "These networks can learn complex patterns from large amounts of data. "
                  "Deep learning powers applications like voice assistants, self-driving cars, and face recognition."
    },
    {
        "id": 6,
        "topic": "deep learning",
        "question": "What is a neural network?",
        "answer": "A neural network is a system of algorithms modeled after the human brain. "
                  "It consists of layers of nodes called neurons that process and pass information. "
                  "Neural networks are used for image recognition, language translation, and more."
    },
    {
        "id": 7,
        "topic": "natural language processing",
        "question": "What is NLP?",
        "answer": "Natural Language Processing or NLP is a branch of AI that helps computers understand, "
                  "interpret, and generate human language. It is used in chatbots, translation apps, "
                  "sentiment analysis, and text summarization."
    },
    {
        "id": 8,
        "topic": "natural language processing",
        "question": "What is a transformer model?",
        "answer": "A transformer is a deep learning model architecture introduced in 2017. "
                  "It uses attention mechanisms to understand the relationship between all words in a sentence. "
                  "Models like BERT, GPT, and T5 are all based on the transformer architecture."
    },
    {
        "id": 9,
        "topic": "artificial intelligence",
        "question": "What is artificial intelligence?",
        "answer": "Artificial intelligence is the simulation of human intelligence in machines. "
                  "AI systems can perform tasks like reasoning, learning, problem solving, and language understanding. "
                  "It includes subfields like machine learning, deep learning, and robotics."
    },
    {
        "id": 10,
        "topic": "artificial intelligence",
        "question": "What is the difference between AI and ML?",
        "answer": "AI is the broader concept of machines performing smart tasks. "
                  "Machine learning is a specific approach within AI where machines learn from data. "
                  "All machine learning is AI but not all AI is machine learning."
    },
]

print(f"✅ Knowledge base ready: {len(knowledge_base)} entries")
print("Topics:", list(set(k["topic"] for k in knowledge_base)))

✅ Knowledge base ready: 10 entries
Topics: ['python', 'machine learning', 'natural language processing', 'deep learning', 'artificial intelligence']


Cell 5 — Load models


In [3]:
# Model 1: FLAN-T5 for answer generation (light and fast)
print("📥 Loading answer generation model (FLAN-T5)...")
gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
gen_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
print("✅ FLAN-T5 ready!")

# Model 2: Sentence embedder for semantic search
print("\n📥 Loading semantic search model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedder ready!")

📥 Loading answer generation model (FLAN-T5)...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder ready!


Cell 6 — FAISS index banao


In [4]:
# Har knowledge entry ko vector mein convert karo
print("Building search index...")

# Question + Answer dono ko embed karo for better matching
texts = [
    f"{k['question']} {k['answer']}"
    for k in knowledge_base
]

embeddings = embedder.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings, dtype="float32")

# FAISS index create karo
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print(f"✅ Search index ready: {index.ntotal} entries indexed!")


def search_knowledge(query, top_k=2):
    """
    Query se most relevant knowledge entries dhundta hai.
    Returns top_k matching entries.
    """
    query_vec  = embedder.encode([query], show_progress_bar=False)
    query_vec  = np.array(query_vec, dtype="float32")
    _, indices = index.search(query_vec, top_k)

    results = []
    for idx in indices[0]:
        results.append(knowledge_base[idx])
    return results


# Quick test
print("\nTest search for 'neural network':")
for r in search_knowledge("neural network", top_k=1):
    print(f"  Found: {r['question']}")

Building search index...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Search index ready: 10 entries indexed!

Test search for 'neural network':
  Found: What is a neural network?


Cell 7 — Answer generator function


In [5]:
def generate_answer(question, context):
    """
    FLAN-T5 se question ka jawab generate karta hai
    retrieved context ke basis par.
    """
    # FLAN-T5 ke liye prompt format
    prompt = (
        f"Answer the following question using the context provided.\n\n"
        f"Context: {context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    output_ids = gen_model.generate(
        inputs["input_ids"],
        max_new_tokens=150,
        num_beams=2,
        early_stopping=True,
        no_repeat_ngram_size=2
    )

    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer.strip()


print("✅ generate_answer() ready!")

# Quick test
test_context = "Python is a high-level programming language used in data science and AI."
test_q       = "What is Python used for?"
print("\nTest answer:")
print(generate_answer(test_q, test_context))

✅ generate_answer() ready!

Test answer:
data science and AI


Cell 8 — Full Q&A agent function


In [6]:
# Chat history store karne ke liye
chat_history = []

def qa_agent(user_question):
    """
    Full Q&A Chat Agent:
    1. User ka question leta hai
    2. Knowledge base mein semantic search karta hai
    3. Relevant context retrieve karta hai
    4. FLAN-T5 se answer generate karta hai
    5. Chat history mein save karta hai
    """

    print(f"\n{'─'*50}")
    print(f"You: {user_question}")
    print(f"{'─'*50}")

    # Step 1: Knowledge base search karo
    relevant = search_knowledge(user_question, top_k=2)

    # Step 2: Context build karo retrieved entries se
    context_parts = []
    for r in relevant:
        context_parts.append(f"{r['question']} {r['answer']}")
    context = " ".join(context_parts)

    # Step 3: Answer generate karo
    answer = generate_answer(user_question, context)

    # Step 4: Source batao
    sources = [r['question'] for r in relevant]

    # Step 5: Chat history mein save karo
    chat_history.append({
        "question": user_question,
        "answer":   answer,
        "sources":  sources
    })

    # Step 6: Print karo
    print(f"Agent: {answer}")
    print(f"\n[Sources matched: {' | '.join(sources)}]")

    return answer


print("✅ qa_agent() ready!")

✅ qa_agent() ready!


Cell 9 — Single question test karo


In [7]:
# Ek question test karo pehle
qa_agent("What is machine learning?")


──────────────────────────────────────────────────
You: What is machine learning?
──────────────────────────────────────────────────
Agent: Machine learning is a type of artificial intelligence

[Sources matched: What is machine learning? | What is supervised learning?]


'Machine learning is a type of artificial intelligence'

Cell 10 — Multiple questions test karo


In [8]:
# Kai questions ek saath test karo
test_questions = [
    "What is Python?",
    "What is the difference between AI and ML?",
    "How does a neural network work?",
    "What is NLP used for?",
    "What is supervised learning?",
]

for q in test_questions:
    qa_agent(q)
    print()


──────────────────────────────────────────────────
You: What is Python?
──────────────────────────────────────────────────
Agent: a high-level, easy-to-learn programming language

[Sources matched: What is Python? | What are Python libraries?]


──────────────────────────────────────────────────
You: What is the difference between AI and ML?
──────────────────────────────────────────────────
Agent: AI is the broader concept of machines performing smart tasks

[Sources matched: What is the difference between AI and ML? | What is artificial intelligence?]


──────────────────────────────────────────────────
You: How does a neural network work?
──────────────────────────────────────────────────
Agent: layers of nodes called neurons that process and pass information

[Sources matched: What is a neural network? | What is deep learning?]


──────────────────────────────────────────────────
You: What is NLP used for?
──────────────────────────────────────────────────
Agent: chatbots, transla

Cell 11 — Chat history dekho


In [9]:
# Puri conversation history print karo
print("=" * 55)
print("CHAT HISTORY")
print("=" * 55)

for i, chat in enumerate(chat_history, 1):
    print(f"\n[{i}] Q: {chat['question']}")
    print(f"    A: {chat['answer']}")

print(f"\nTotal questions asked: {len(chat_history)}")

CHAT HISTORY

[1] Q: What is machine learning?
    A: Machine learning is a type of artificial intelligence

[2] Q: What is Python?
    A: a high-level, easy-to-learn programming language

[3] Q: What is the difference between AI and ML?
    A: AI is the broader concept of machines performing smart tasks

[4] Q: How does a neural network work?
    A: layers of nodes called neurons that process and pass information

[5] Q: What is NLP used for?
    A: chatbots, translation apps, sentiment analysis, and text summarization

[6] Q: What is supervised learning?
    A: a machine learning technique where the model is trained on labeled data

Total questions asked: 6


Cell 12 — Live interactive chat loop


In [ ]:
# Is cell mein real-time chat kar sakte ho
# Type karo aur Enter dabaao
# 'quit' likhne par band ho jaayega

print("🤖 Q&A Chat Agent ready!")
print("Kuch bhi poochho. 'quit' likhne par band hoga.\n")
print("Topics: Python | Machine Learning | Deep Learning | NLP | AI")
print("=" * 55)

while True:
    try:
        user_input = input("\nTum: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ["quit", "exit", "band karo", "q"]:
            print("\nAgent: Theek hai! Alvida! 👋")
            break

        if user_input.lower() in ["history", "chat history"]:
            print(f"\nAbhi tak {len(chat_history)} sawaal pooche gaye.")
            for i, c in enumerate(chat_history[-3:], 1):
                print(f"  {i}. {c['question']}")
            continue

        if user_input.lower() in ["topics", "topics kya hain"]:
            topics = list(set(k["topic"] for k in knowledge_base))
            print("Available topics:", ", ".join(topics))
            continue

        qa_agent(user_input)

    except KeyboardInterrupt:
        print("\n\nAgent: Bye bye! 👋")
        break

🤖 Q&A Chat Agent ready!
Kuch bhi poochho. 'quit' likhne par band hoga.

Topics: Python | Machine Learning | Deep Learning | NLP | AI

Tum: ai

──────────────────────────────────────────────────
You: ai
──────────────────────────────────────────────────
Agent: is the simulation of human intelligence in machines

[Sources matched: What is artificial intelligence? | What is the difference between AI and ML?]

Tum: Artificial Intelligence (AI) is the development of computer systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and understanding language

──────────────────────────────────────────────────
You: Artificial Intelligence (AI) is the development of computer systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and understanding language
──────────────────────────────────────────────────
Agent: human intelligence in machines

[Sources match

Cell 13 — Apna knowledge add karo


In [ ]:
# Apni khud ki Q&A entry add karo
# Aur index rebuild karo

def add_to_knowledge(topic, question, answer):
    """
    Knowledge base mein naya entry add karta hai
    aur FAISS index rebuild karta hai.
    """
    new_entry = {
        "id":       len(knowledge_base) + 1,
        "topic":    topic,
        "question": question,
        "answer":   answer
    }
    knowledge_base.append(new_entry)

    # Index rebuild karo naye entry ke saath
    texts = [f"{k['question']} {k['answer']}" for k in knowledge_base]
    new_embeddings = embedder.encode(texts, show_progress_bar=False)
    new_embeddings = np.array(new_embeddings, dtype="float32")

    index.reset()
    index.add(new_embeddings)

    print(f"✅ Added: '{question}'")
    print(f"   Total entries: {len(knowledge_base)}")


# Example — apna entry add karo:
add_to_knowledge(
    topic    = "python",
    question = "What is a Python decorator?",
    answer   = "A decorator is a function that modifies another function's behavior. "
               "It is written with @ symbol above a function definition. "
               "Decorators are used for logging, authentication, and caching."
)

# Test karo naya entry
qa_agent("What is a Python decorator?")